# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [82]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR
    
RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: g:\내 드라이브\dev_GD\llm-data-analysis-study\notebooks
프로젝트 루트: g:\내 드라이브\dev_GD\llm-data-analysis-study
원본 데이터 폴더: g:\내 드라이브\dev_GD\llm-data-analysis-study\practice\chapter05\data\raw
전처리 데이터 폴더: g:\내 드라이브\dev_GD\llm-data-analysis-study\data\processed
보고서 폴더: g:\내 드라이브\dev_GD\llm-data-analysis-study\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [83]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [84]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [85]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [86]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [87]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [88]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [89]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [90]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [91]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [92]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [93]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_29636\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_29636\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

In [94]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [95]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [96]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [97]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [98]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [99]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [100]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [101]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [102]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [103]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [104]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [105]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [106]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/data/processed/customers_clean.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/data/processed/products_clean.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/data/processed/orders_clean.csv'),
 WindowsPath('g:/내 드라이브/dev_GD/llm-data-analysis-study/data/processed/order_items_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [107]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: g:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [108]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [109]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_29636\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_29636\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [110]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [111]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [112]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


### LLM 전처리 코드 검토 결과

- `customers["age"]`: 원본 DataFrame을 직접 수정하므로 복사본에서 처리하는 것이 적절하다. 또한 숫자 변환 여부와 데이터 분포를 확인하지 않고 평균값으로 대체하는 것은 주의가 필요하다.
- `drop_duplicates()`: 완전 중복 행은 제거할 수 있지만, 주요 ID의 중복 여부는 별도로 확인해야 한다.
- `orders["order_date"]`: 날짜 변환에 실패할 가능성이 있으므로 `errors="coerce"`를 사용하고 `NaT` 건수를 확인할 필요가 있다.
- `order_items["line_total"]`: 계산 전에 `quantity`와 `unit_price`가 숫자형인지 확인해야 한다.

이번 실습에서는 원본 데이터를 보존하고, 타입 변환 실패와 중복 여부를 확인한 뒤 처리 결과를 재검증하는 방식이 더 적절하다고 판단하였다.


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [113]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.
missing_tables = []

for name, df in processed_data.items():
    summary = missing_summary(df).reset_index()
    summary = summary.rename(columns={'index': 'column'})
    summary['dataset'] = name
    missing_tables.append(summary)

all_missing = pd.concat(missing_tables, ignore_index=True)

all_missing = all_missing[
    ['dataset', 'column', 'missing_count', 'missing_ratio']
]

display(all_missing)

,dataset,column,missing_count,missing_ratio
0,customers,signup_date,2,1.33
1,customers,customer_id,0,0.00
2,customers,name,0,0.00
3,customers,gender,0,0.00
4,customers,age,0,0.00
5,customers,city,0,0.00
6,products,product_id,0,0.00
7,products,product_name,0,0.00
8,products,category,0,0.00
9,products,price,0,0.00


In [114]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
age_outliers = customers_clean[
    (customers_clean['age'] < 18) |
    (customers_clean['age'] > 100)
]

display(age_outliers[['customer_id', 'age']])
print('age 이상값 후보:', len(age_outliers))

,customer_id,age
32,33,150.0


age 이상값 후보: 1


In [115]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
monthly_counts = (
    orders_clean.dropna(subset=['order_date'])
    .groupby('order_month')
    .size()
    .reset_index(name='order_count')
    .sort_values('order_month')
)

display(monthly_counts)

print('집계 대상 주문 건수:', monthly_counts['order_count'].sum())
print('order_date 결측 건수:', orders_clean['order_date'].isna().sum())
print('※ 집계 기간 내 표에 표시되지 않은 월은 주문 건수가 0건입니다.')

,order_month,order_count
0,2025-07,19
1,2025-08,26
2,2025-09,12
3,2025-10,22
4,2025-11,23
5,2025-12,23
6,2026-01,26
7,2026-02,19
8,2026-03,22
9,2026-04,26


집계 대상 주문 건수: 297
order_date 결측 건수: 3
※ 집계 기간 내 표에 표시되지 않은 월은 주문 건수가 0건입니다.


In [116]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
top10_line_total = (
    order_items_clean
    .sort_values('line_total', ascending=False)
    .head(10)
)

display(top10_line_total[
    ['order_item_id', 'order_id', 'product_id',
     'quantity', 'unit_price', 'line_total']
])

,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
9,10,5,65,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
116,117,51,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
491,492,195,99,5.0,187000.0,935000.0


### 과제 5. 전처리 전후 행 수 변화

전처리 결과 네 개 dataset 모두 행 수가 감소하였다.

- `customers`: 152 → 150건 (완전 중복 2건 제거)
- `products`: 102 → 96건 (완전 중복 2건 제거, `price` 변환 실패 및 0 이하 값 4건 제외)
- `orders`: 302 → 300건 (완전 중복 2건 제거)
- `order_items`: 765 → 752건 (완전 중복 2건 제거, `quantity`와 `unit_price` 변환 실패 및 0 이하 값 11건 제외)

전체 행 수는 1,321건에서 1,298건으로 23건 감소하였다.

행 수 감소는 이번 실습에서 적용한 중복 및 이상값 처리 기준에 따른 결과이다.  
다만 제외된 행이 실제 오류인지 확인한 것은 아니므로
전처리 이후 데이터 간 참조관계와 정보 손실 가능성을 함께 검토할 필요가 있다.

### 과제 6. LLM을 활용한 결측치 처리 기준 검토

온라인 쇼핑몰의 `customers`, `products`, `orders`, `order_items` 데이터를 전처리하고 있습니다.
실제 데이터를 확인한 결과 다음과 같은 결측치 및 타입 변환 문제가 발견되었습니다.

- `customers.age`: 원본 결측 4건, 숫자 변환 실패 2건
- `customers.city`: 결측 2건
- `customers.signup_date`: 날짜 변환 후 `NaT` 2건
- `products.price`: 숫자 변환 실패 2건
- `orders.order_date`: 날짜 변환 후 `NaT` 3건
- `order_items.quantity`: 숫자 변환 실패 5건
- `order_items.unit_price`: 숫자 변환 실패 2건

현재 `age`는 중앙값 41로 대체하고 `city`는 `Unknown`으로 처리했습니다. `price`, `quantity`, `unit_price`는 숫자로 변환할 수 없거나 0 이하인 행을 제외했으며 날짜 변환 후 발생한 `NaT`는 그대로 유지하고 있습니다.

이 전처리 기준이 적절한지 검토해 주세요.

1. 각 column의 결측치 및 변환 실패값에 대해 삭제, 대체, 유지 중 어떤 방법이 적절한지 비교해 주세요.
2. 현재 적용한 처리 방법으로 인해 정보 손실이나 데이터 분포의 왜곡이 발생할 가능성이 있는지 설명해 주세요.
3. `products`와 `order_items`에서 행을 제외한 것이 PK/FK 관계나 이후 분석에 미칠 영향을 검토해 주세요.
4. 전처리 후 추가로 확인해야 할 검증 항목을 제안해 주세요.

원본 결측치, 타입 변환 실패, 0 이하의 이상값 후보를 구분해서 검토해 주시고, 실제 업무적 의미가 확인되지 않은 값에 대해서는 임의로 오류라고 단정하지 말아 주세요.




# Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

> 주 제출물은 실행 완료 Notebook `chapter05/chapter05.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름:김희철
- GitHub ID:KHeeCheol
- 작성일:2026-10-13
- 최종 제출 URL:https://github.com/KHeeCheol/llm-data-analysis-study/blob/main/notebooks/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape:
  - customers: (152, 6)
  - products: (102, 4)
  - orders: (302, 5)
  - order_items: (765, 5) 

- 결측치:
  - customers: age 4건(2.63%), city 2건(1.32%)
  - products: 0건
  - orders: 0건
  - order_items: 0건

- 전체 중복:
  - customers: 2건
  - products: 2건
  - orders: 2건
  - order_items: 2건

- 주요 ID 중복:
  - customer_id: 2건
  - product_id: 2건
  - order_id: 2건
  - order_item_id: 2건

- 타입 문제 후보:
  - age, price, quantity, unit_price: 숫자로 변환할 수 없는 문자열 포함
  - order_date, signup_date: 날짜형 변환 필요

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
원본 데이터에서는 고객 나이와 도시 정보에 결측치가 있었으며 나머지 세 데이터에서는 결측치가 확인되지 않았다. 네 데이터 모두 완전 중복 행이 2건씩 존재했고 주요 ID의 중복 건수도 동일하게 나타났다.

### 나의 해석과 판단
전처리 전에 결측치와 중복 여부를 먼저 확인하고 이후 타입 변환이나 데이터 정리 과정에서 발생하는 변화를 구분할 필요가 있다고 판단하였다.
특히 결측치가 없더라도 숫자 컬럼에 문자열이 포함되어 있을 수 있으므로 결측치 검사만으로 데이터 품질을 판단하기는 어렵다. 또한 전체 행 중복과 ID 중복은 의미가 다르므로 구분해서 확인해야 한다.

### 업무·분석적 의미
전처리 전 상태를 기록해 두면 이후 데이터의 행 수나 결측치가 변경되었을 때 그 원인을 확인할 수 있다. 특히 여러 데이터를 연결하는 분석에서는 주요 ID의 중복 여부가 데이터 병합 결과에 영향을 줄 수 있다.

### 한계와 추가 확인 사항
현재 검사만으로는 숫자·날짜 변환 문제나 비정상적인 값까지 확인할 수 없다. 또한 각 데이터의 ID가 다른 데이터와 정상적으로 연결되는지는 별도 검증이 필요하다.

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리:
  - 문자열 컬럼의 앞뒤 공백을 제거하였다.
  - 도시명처럼 공백 차이로 구분될 수 있는 값을 정리하였다.
  - 주문 상태는 공백 제거 이후에도 대소문자 및 한글 표기 차이가 남아 있어 추가적인 표준화가 필요하다.

- 숫자 변환 대상과 실패 건수:
  - customers.age: 2건
  - products.price: 2건
  - order_items.quantity: 5건
  - order_items.unit_price: 2건

- 날짜 변환 대상과 실패 건수:
  - orders.order_date: datetime 변환 후 NaT 3건
  - customers.signup_date: datetime 변환 후 NaT 2건
  - 변환 후 주문일 범위: 2025-07-01 ~ 2028-12-31

- 결측 처리 기준:
  - age: 원본 결측 4건과 숫자 변환 실패 2건을 중앙값 41세로 대체
  - city: 결측치 2건을 'Unknown'으로 대체
  - 가격·수량·단가·날짜 변환에 실패한 값은 임의로 대체하지 않고 추가 검토 대상으로 유지

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
고객 나이는 기존 결측치 4건 외에 숫자 변환에 실패한 값이 2건 추가로 확인되었다. 총 6건을 중앙값 41세로 대체한 결과로 나이 컬럼의 결측치는 0건이 되었다. 도시 정보의 결측치 2건도 'Unknown'으로 처리하였다.  
상품 가격과 주문 상세의 수량 및 단가에서도 숫자 변환 실패가 확인되었다. 날짜 변환 후에는 주문일 3건, 가입일 2건이 NaT로 나타났으며, 주문일의 최대값이 2028년 12월 31일로 확인되었다.

### 나의 해석과 판단
나이가 누락된 고객을 삭제하면 해당 고객의 다른 정보까지 함께 제외되므로 이번 실습에서는 중앙값 대체를 선택하였다. 중앙값은 극단적인 값의 영향을 평균보다 적게 받지만 실제 나이를 알 수 없는 값을 동일한 값으로 채운다는 점에서 한계가 있다.  
도시 정보는 특정 지역으로 임의 대체하기보다 'Unknown'으로 남기는 것이 적절하다고 판단하였다. 가격이나 수량처럼 실제 거래 계산에 사용되는 값은 근거 없이 대체할 경우 분석 결과가 달라질 수 있으므로 추가 확인이 필요하다.

### 업무·분석적 의미
문자열과 데이터 타입을 정리해야 동일한 범주를 정확히 집계하고 금액 계산이나 날짜별 분석을 수행할 수 있다. 특히 타입 변환에 실패한 값을 확인하지 않으면 일부 데이터가 계산에서 누락되더라도 그 원인을 파악하기 어려울 수 있다.

### 한계와 추가 확인 사항
중앙값으로 대체한 나이는 실제 관측값이 아니므로 연령 분포에 영향을 줄 수 있다. 또한 숫자 및 날짜 변환에 실패한 값은 아직 원인이나 업무적 의미를 확인하지 않았다.  
주문일에서 확인된 2028년 날짜 역시 형식상 정상적인 날짜이지만 실제 주문 데이터로 적절한지는 추가 검토가 필요하다.

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복:
  - customers, products, orders, order_items에서 각각 2건씩 총 8건의 완전 중복 행 제거
  - customer_id, product_id, order_id, order_item_id의 중복 여부 확인
  - order_items.order_id처럼 관계상 반복될 수 있는 값은 중복 제거 대상으로 보지 않음

- 표준화한 범주값:
  - order_status의 영문 대소문자와 한글 표기를 통일하였다.
  - completed 180건, cancelled 63건, refunded 57건으로 정리되었다.
  
- 허용값 밖 값:
  - order_status: 표준화 이후 허용 범주 외 값 0건
  - products.price <= 0: 2건
  - order_items.quantity <= 0: 3건
  - order_items.unit_price <= 0: 1건

- 이상값 후보:
  - products.price 최댓값: 9,999,999
  - order_items.quantity 최댓값: 100
  - 숫자 변환 실패: price 2건, quantity 5건, unit_price 2건

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
네 개의 dataset에서 각각 2건씩 완전 중복 행이 확인되어 총 8건을 제거하였다. order_status는 표준화 이후 completed, cancelled, refunded 세 가지 값으로 정리되었으며, orders의 전체 300건은 유지되었다.  

숫자형 column에서는 0 이하 값과 일부 극단적인 값이 확인되었다. 실습에서 정한 기준에 따라 price, quantity, unit_price가 0 이하이거나 숫자로 변환되지 않은 행을 제외한 결과 products는 100건에서 96건으로 order_items는 763건에서 752건으로 감소하였다.

### 나의 해석과 판단
모든 column 값이 동일한 행은 중복 저장된 것으로 판단하여 제거하였다. 다만 order_id처럼 다른 dataset과의 관계에서 반복될 수 있는 값은 중복이라는 이유만으로 제거해서는 안 된다고 판단하였다.  

price, quantity, unit_price가 0 이하인 값은 이번 실습 기준에 따라 제외하였다. 하지만 실제 업무에서는 price = 0이 증정품을 의미하거나 quantity < 0이 반품을 의미할 가능성도 있으므로 자동으로 삭제하는 것이 항상 적절하지는 않다.  

또한 price = 9999999나 quantity = 100은 다른 값들과 차이가 크지만, 실제 거래일 가능성을 배제할 수 없어 이번 단계에서는 이상값 후보로만 확인하였다.

### 업무·분석적 의미
완전 중복 행을 제거하면 동일한 데이터가 중복 집계되는 문제를 줄일 수 있다. 또한 order_status를 표준화하면 같은 의미의 값이 서로 다른 범주로 집계되는 문제를 방지할 수 있다.  

반면 price, quantity, unit_price를 기준으로 행을 제외하면 분석 대상이 줄어든다. 특히 products에서 일부 product_id가 제외되면 해당 값을 참조하는 order_items.product_id와의 관계에도 영향을 줄 수 있으므로 후속 검증이 필요하다.

### 한계와 추가 확인 사항
이번 이상값 처리 기준은 실습을 위한 것으로 실제 업무 데이터에 그대로 적용하기는 어렵다. 제외된 값에 증정품이나 반품 등의 의미가 포함되어 있다면 정보 손실이 발생할 수 있다.  

또한 최댓값으로 확인된 products.price와 order_items.quantity의 실제 의미는 확인하지 못했으므로 이후 분석에서 해당 값이 결과에 미치는 영향을 추가로 검토할 필요가 있다.

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: orders.customer_id에서 2건
- 없는 `order_id`: order_items.order_id에서 2건
- 없는 `product_id`: order_items.product_id에서 26건
- 생성한 파생 컬럼: order_month, order_dayofweek, line_total
- `line_total` 검증:
  - quantity × unit_price로 계산
  - 상위 5개 행의 계산 결과 확인
  - 전체 line_total 합계: 252,140,000원

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
order_items에 line_total을 생성하고 계산 결과를 확인하였다. 이후 전처리된 데이터의 PK/FK 관계를 검사한 결과, 참조 대상이 없는 customer_id와 order_id가 각각 2건, product_id가 26건 확인되었다.  

특히 product_id의 불일치가 상대적으로 많았으며 이는 products의 일부 행을 제외한 과정과 관련이 있을 수 있다. 또한 line_total의 합계는 252,140,000원이었지만 order_status나 참조관계 오류를 반영한 금액은 아니다.
### 나의 해석과 판단
전처리 전에는 정상적으로 연결되던 데이터도 특정 행이 제외되면서 참조관계가 달라질 수 있으므로 PK/FK 검증을 다시 수행할 필요가 있다고 판단하였다.  
특히 product_id가 일치하지 않는 26건을 바로 삭제하기보다는 원본부터 존재하지 않았던 값인지 전처리 과정에서 발생한 문제인지 먼저 구분해야 한다.  
또한 line_total은 order_items의 각 행에서 계산한 금액이므로, 전체 합계를 실제 매출액으로 해석하기에는 한계가 있다.

## 5. 전처리 전/후 비교
| 항목    |  처리 전 |  처리 후 | 변화 이유                                              |
| ----- | ----: | ----: | -------------------------------------------------- |
| 행 수   | 1,321 | 1,298 | 완전 중복 8건 및 숫자 변환 실패·0 이하 값에 해당하는 15건 제외            |
| 결측    |     6 |    11 | 기존 결측 처리 후 날짜 변환 실패와 파생 컬럼의 결측 반영                  |
| 중복    |     8 |     0 | 각 dataset에서 완전 중복 행 2건씩 제거                         |
| 변환 실패 |    16 |     5 | 숫자 변환 실패값은 대체하거나 해당 행을 제외했으며, 날짜 변환 후 `NaT` 5건은 유지 |



![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
전처리 결과 전체 행 수는 1,321건에서 1,298건으로 23건 감소하였다. customers와 orders는 완전 중복 행이 각각 2건씩 제거되었으며, products는 102건에서 96건으로, order_items는 765건에서 752건으로 감소하였다.

전체 중복 행은 8건에서 0건으로 줄었지만 결측 셀 수는 6개에서 11개로 증가하였다. 이는 signup_date와 order_date에서 날짜 변환에 실패한 값이 남아 있고 order_date를 이용해 생성한 order_month, order_dayofweek에도 결측이 발생했기 때문이다.

### 나의 해석과 판단
행 수 감소는 완전 중복 제거와 price, quantity, unit_price에 적용한 처리 기준에 따른 것으로 확인하였다. 따라서 이번 실습에서 의도한 전처리 결과로 판단하였다.  
결측치가 증가한 부분도 단순히 전처리 품질이 나빠졌다고 볼 수는 없다. 원본에서 정상적인 문자열처럼 저장되어 있던 값이 타입 변환 과정에서 실제 결측으로 드러났기 때문이다.  
다만 products에서 일부 행을 제외한 뒤 order_items.product_id의 참조관계 문제가 확인되었으므로 행 수가 줄었다는 사실만으로 전처리가 적절했다고 판단하기는 어렵다.

### 업무·분석적 의미
전처리 전후의 행 수와 결측치, 중복 상태를 비교하면 어떤 처리로 데이터가 변경되었는지 확인할 수 있다. 특히 숫자나 날짜 변환으로 새롭게 발생한 결측치는 후속 분석에서 데이터가 누락되는 원인이 될 수 있으므로 별도로 관리할 필요가 있다.  
또한 특정 조건에 따라 행을 제외하면 분석 대상뿐 아니라 다른 dataset과의 연결관계도 달라질 수 있다는 점을 확인하였다.

### 한계와 추가 확인 사항
이번 실습에서는 숫자 변환 실패와 0 이하 값을 정해진 기준에 따라 제외했지만, 실제 업무에서는 해당 값의 의미를 먼저 확인해야 한다.  
order_date와 signup_date에 남아 있는 NaT는 아직 처리하지 않았으며 분석 목적에 따라 제외 또는 보류 여부를 결정할 필요가 있다. 전처리 과정에서 제외된 행과 PK/FK 불일치가 이후 분석 결과에 미치는 영향도 추가로 확인해야 한다.

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV:
  - data/processed/customers_clean.csv
  - data/processed/products_clean.csv
  - data/processed/orders_clean.csv
  - data/processed/order_items_clean.csv

- 생성된 요약 보고서:
  - reports/ch05_preprocessing_summary.md

- `python scripts/preprocess_data.py` 재실행 결과:
  - 직접 실행 시 ModuleNotFoundError: No module named 'src' 오류 발생
  - 프로젝트 루트에서 python -m scripts.preprocess_data로 실행하여 정상 완료
  - 전처리 전후 shape, 중복 검사, PK/FK 검사 결과 확인
  - clean CSV 4개와 요약 보고서 생성 확인
- 재실행 후 달라진 점:
  - customers: (150, 6)
  - products: (96, 4)
  - orders: (300, 7)
  - order_items: (752, 6)
  - Notebook에서 얻은 결과와 동일한 shape가 확인되었으며, 전체 중복 및 주요 ID 중복도 모두 0건으로 나타남
  - PK/FK 검증 결과 역시 기존과 동일하게 2건, 2건, 26건으로 확인됨
  - 저장된 CSV를 다시 불러온 결과, shape와 column 구성이 일치하였고 요약 보고서도 정상적으로 존재함

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것의 차이를 작성하세요.
Notebook에서는 각 셀의 실행 순서나 변수 상태에 따라 결과가 달라질 수 있지만 스크립트는 정해진 전처리 과정을 처음부터 일관되게 실행할 수 있다는 차이가 있다.  

이번 실습에서는 Notebook에서 개별적으로 수행한 전처리를 함수와 스크립트로 다시 실행하였으며 데이터 크기와 중복, PK/FK 검증 결과가 일치하는 것을 확인하였다.  

또한 CSV를 저장한 후 다시 불러와 구조를 확인함으로써 코드 실행뿐 아니라 결과 파일이 정상적으로 생성되었는지도 검증할 수 있었다.  

다만 현재는 주요 검증 지표가 일치하는 수준까지 확인한 것이므로, 모든 데이터 값이 완전히 동일한지까지 검증한 것은 아니다.
## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 원본 데이터 보존 및 변경 내역 확인
원본 데이터는 직접 수정하지 않고 별도의 DataFrame에서 전처리를 진행한다. 또한 처리 전후의 행 수, 결측치, 중복 등의 변화를 기록하여 어떤 작업으로 데이터가 변경되었는지 확인한다.

2. 데이터의 의미를 고려한 처리 기준 적용
결측치나 이상값 후보가 발견되더라도 일괄적으로 삭제하거나 대체하지 않는다. 특히 price, quantity, unit_price처럼 실제 거래와 관련된 값은 업무상 의미를 고려하여 처리 여부를 결정할 필요가 있다.

3. 전처리 후 검증 및 재현성 확보
전처리 결과는 데이터 크기와 값의 변화뿐 아니라 PK/FK 관계까지 다시 확인한다. 또한 동일한 원본에 대해 전처리 과정을 반복 실행했을 때 일관된 결과가 나오는지 검증한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
이번 실습에서 가장 주의해야 할 처리는 price, quantity, unit_price가 0 이하이거나 숫자로 변환되지 않은 행을 일괄 제외하는 과정이라고 생각한다.  

실습에서는 해당 조건으로 products 4건과 order_items 11건을 추가로 제외하였다. 그러나 price = 0은 증정품, quantity < 0은 반품을 의미할 가능성이 있어 실제 업무에서는 모두 오류라고 판단하기 어렵다.  

또한 전처리 후 order_items.product_id와 products.product_id 사이에 26건의 참조 불일치가 확인되었다. 이 불일치가 모두 이번 행 제외로 발생했다고 볼 수는 없지만, 한 dataset에서 행을 제외하는 작업이 다른 dataset과의 관계에도 영향을 줄 수 있다는 점을 확인하였다.  

따라서 자동으로 행을 삭제하기보다 해당 값의 의미와 다른 데이터에 미치는 영향을 확인한 후 처리하는 것이 적절하다고 판단하였다.
### 다음 EDA에서 특히 주의할 데이터 특성
orders.order_status에는 completed, cancelled, refunded가 함께 포함되어 있으므로 거래금액을 분석할 때 어떤 주문 상태를 집계 대상으로 사용할지 먼저 결정해야 한다. 특히 order_items.line_total의 단순 합계를 실제 매출액으로 해석해서는 안 된다.  

또한 orders.order_date의 결측 3건과 2028년 날짜는 월별 분석에 영향을 줄 수 있다. order_items.quantity = 100이나 products.price = 9999999처럼 다른 값에 비해 큰 데이터도 평균이나 금액 합계에 영향을 줄 수 있으므로 주의가 필요하다.  

PK/FK 불일치가 남아 있다는 점도 고려해야 한다. 여러 dataset을 결합할 경우 일부 행이 누락되거나 집계 결과가 달라질 가능성이 있다.
### 현재 전처리의 한계
이번 전처리는 교재에서 제시한 기준에 따라 수행했으며 개별 데이터의 실제 업무적 의미까지 확인한 것은 아니다. 따라서 일부 제외된 행에 정상적인 거래나 반품 정보가 포함되어 있을 가능성이 있다.  

또한 customers.age의 결측치를 중앙값으로 대체했지만 age = 150과 같은 이상값 후보는 그대로 남아 있다. 날짜 변환에 실패한 값과 PK/FK 불일치 역시 완전히 해결하지 못했다.  

Notebook과 스크립트의 주요 검증 결과는 일치했으나 모든 데이터 값을 개별적으로 비교한 것은 아니다. 향후 실제 데이터를 분석할 때에는 이러한 한계를 고려하여 전처리 기준을 다시 검토할 필요가 있다.
## 최종 제출 체크
- [O] raw 원본을 덮어쓰지 않았습니다.
- [O] 처리 전/후 비교를 남겼습니다.
- [O] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [O] clean CSV와 재실행 결과를 확인했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.
